# Modular scRNA-seq Notebook

This notebook is organized by modules:
1. config
2. data I/O
3. preprocessing
4. dimensionality reduction (PCA/UMAP)
5. autoencoder (optional)
6. outputs

By default, AE training is off.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# ----- Paths -----
COUNTS_CSV = Path('/proj/daiweilab/users/jinweizh/735/project_data/data copy/brain_counts.csv')
META_CSV = Path('/proj/daiweilab/users/jinweizh/735/project_data/data copy/brain_metadata.csv')
OUTDIR = Path('/proj/daiweilab/users/jinweizh/735/project_data/notebook_outputs')
OUTDIR.mkdir(parents=True, exist_ok=True)

# ----- Columns -----
CELL_COL = 'cell'
LABEL_COL = 'cell_ontology_class'

# ----- Preprocessing -----
SEED = 42
TARGET_SUM = 1e4
N_HVG = 2000

# ----- PCA/UMAP -----
N_PCS = 30
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.3

# ----- Autoencoder (optional) -----
RUN_AE_TRAINING = False
LATENT_DIM = 16
HIDDEN_DIMS = [512, 128]
EPOCHS = 100
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-5
VAL_FRAC = 0.1
PATIENCE = 15

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Module A: Utility Functions

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def plot_embedding(emb: np.ndarray, labels: pd.Series, title: str, xlab: str, ylab: str):
    plt.figure(figsize=(8, 6))
    arr = labels.astype(str).values
    for lab in pd.unique(arr):
        idx = arr == lab
        plt.scatter(emb[idx, 0], emb[idx, 1], s=10, alpha=0.8, label=lab)
    plt.title(title)
    plt.xlabel(xlab)
    plt.ylabel(ylab)
    plt.legend(loc='best', fontsize=8, frameon=False)
    plt.tight_layout()
    plt.show()

## Module B: Data I/O

In [ ]:
def load_sc_data(counts_csv: Path, meta_csv: Path, cell_col: str, label_col: str):
    counts_df = pd.read_csv(counts_csv)
    meta_df = pd.read_csv(meta_csv)

    if counts_df.shape[1] < 2:
        raise ValueError('Counts CSV must contain cell column + gene columns.')
    if cell_col not in meta_df.columns or label_col not in meta_df.columns:
        raise ValueError('Metadata is missing required columns.')

    cell_ids = counts_df.iloc[:, 0].astype(str)
    gene_names = counts_df.columns[1:].to_numpy()
    expr = counts_df.iloc[:, 1:].to_numpy(dtype=np.float32)

    meta_df[cell_col] = meta_df[cell_col].astype(str)
    meta_df = meta_df.set_index(cell_col)
    if not pd.Index(cell_ids).isin(meta_df.index).all():
        raise ValueError('Some cells in counts are missing in metadata.')
    labels = meta_df.loc[cell_ids, label_col]

    return {
        'cell_ids': cell_ids,
        'gene_names': gene_names,
        'expr': expr,
        'labels': labels,
    }

## Module C: Preprocessing

In [ ]:
def library_normalize_log1p(expr: np.ndarray, target_sum: float):
    libsize = expr.sum(axis=1, keepdims=True)
    libsize[libsize == 0] = 1.0
    expr_norm = (expr / libsize) * target_sum
    return np.log1p(expr_norm).astype(np.float32)

def select_hvg(log_expr: np.ndarray, gene_names: np.ndarray, n_hvg: int):
    means = log_expr.mean(axis=0)
    vars_ = log_expr.var(axis=0)
    disp = vars_ / (means + 1e-8)
    valid = np.isfinite(disp) & (means > 0)
    valid_idx = np.where(valid)[0]
    if valid_idx.size == 0:
        raise ValueError('No valid genes for HVG selection.')
    keep = min(n_hvg, valid_idx.size)
    top_local = np.argpartition(disp[valid], -keep)[-keep:]
    top_idx = valid_idx[top_local]
    top_idx = top_idx[np.argsort(disp[top_idx])[::-1]]
    return log_expr[:, top_idx], gene_names[top_idx]

def standardize_by_train(x: np.ndarray, train_idx: np.ndarray):
    x_mean = x[train_idx].mean(axis=0, keepdims=True)
    x_std = x[train_idx].std(axis=0, keepdims=True)
    x_std[x_std < 1e-6] = 1.0
    x_scaled = (x - x_mean) / x_std
    return x_scaled, x_mean, x_std

## Module D: Dimensionality Reduction

In [ ]:
def run_pca(expr_hvg: np.ndarray, n_pcs: int, seed: int):
    n = min(n_pcs, expr_hvg.shape[0] - 1, expr_hvg.shape[1])
    if n < 2:
        raise ValueError('Not enough dimensions for PCA.')
    model = PCA(n_components=n, random_state=seed)
    emb = model.fit_transform(expr_hvg)
    return model, emb

def run_umap(pca_emb: np.ndarray, seed: int, n_neighbors: int, min_dist: float):
    model = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric='euclidean',
        random_state=seed,
    )
    emb = model.fit_transform(pca_emb)
    return model, emb

def save_pca_umap_outputs(outdir: Path, cell_ids: pd.Series, labels: pd.Series, pca_model, pca_emb, umap_emb, hvg_names):
    pca_df = pd.DataFrame(pca_emb, columns=[f'PC{i+1}' for i in range(pca_emb.shape[1])])
    pca_df.insert(0, 'cell', cell_ids.values)
    pca_df['label'] = labels.values
    pca_df.to_csv(outdir / 'pca_embedding.csv', index=False)

    umap_df = pd.DataFrame({
        'cell': cell_ids.values,
        'label': labels.values,
        'UMAP1': umap_emb[:, 0],
        'UMAP2': umap_emb[:, 1],
    })
    umap_df.to_csv(outdir / 'umap_embedding.csv', index=False)

    pd.Series(hvg_names, name='gene').to_csv(outdir / 'hvg_genes.csv', index=False)
    pd.DataFrame({
        'PC': [f'PC{i+1}' for i in range(len(pca_model.explained_variance_ratio_))],
        'explained_variance_ratio': pca_model.explained_variance_ratio_,
    }).to_csv(outdir / 'pca_explained_variance_ratio.csv', index=False)

## Run PCA + UMAP Pipeline

In [ ]:
set_seed(SEED)

data = load_sc_data(COUNTS_CSV, META_CSV, CELL_COL, LABEL_COL)
cell_ids = data['cell_ids']
gene_names = data['gene_names']
expr = data['expr']
labels = data['labels']

expr_log = library_normalize_log1p(expr, TARGET_SUM)
expr_hvg, hvg_names = select_hvg(expr_log, gene_names, N_HVG)

pca_model, pca_emb = run_pca(expr_hvg, N_PCS, SEED)
_, umap_emb = run_umap(pca_emb, SEED, UMAP_NEIGHBORS, UMAP_MIN_DIST)

save_pca_umap_outputs(OUTDIR, cell_ids, labels, pca_model, pca_emb, umap_emb, hvg_names)

print('Cells:', expr.shape[0], 'Genes:', expr.shape[1], 'HVG:', expr_hvg.shape[1])
plot_embedding(pca_emb[:, :2], labels, 'PCA (first 2 components)', 'PC1', 'PC2')
plot_embedding(umap_emb, labels, 'UMAP (from PCA space)', 'UMAP1', 'UMAP2')

## Module E: Autoencoder (Optional)

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], latent_dim: int):
        super().__init__()
        enc = []
        d = input_dim
        for h in hidden_dims:
            enc.extend([nn.Linear(d, h), nn.ReLU()])
            d = h
        enc.append(nn.Linear(d, latent_dim))
        self.encoder = nn.Sequential(*enc)

        dec = []
        d = latent_dim
        for h in reversed(hidden_dims):
            dec.extend([nn.Linear(d, h), nn.ReLU()])
            d = h
        dec.append(nn.Linear(d, input_dim))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

def run_epoch(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, total_n = 0.0, 0
    for (x,) in loader:
        x = x.to(device, non_blocking=True)
        x_hat, _ = model(x)
        loss = criterion(x_hat, x)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)
    return total_loss / max(total_n, 1)

def run_autoencoder(expr_hvg: np.ndarray, labels: pd.Series):
    idx_all = np.arange(expr_hvg.shape[0])
    idx_train, idx_val = train_test_split(
        idx_all,
        test_size=VAL_FRAC,
        random_state=SEED,
        shuffle=True,
        stratify=labels.values,
    )

    x_scaled, x_mean, x_std = standardize_by_train(expr_hvg, idx_train)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(x_scaled[idx_train].astype(np.float32))), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.from_numpy(x_scaled[idx_val].astype(np.float32))), batch_size=BATCH_SIZE, shuffle=False)

    model = Autoencoder(x_scaled.shape[1], HIDDEN_DIMS, LATENT_DIM).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val = float('inf')
    best_state = None
    bad_epochs = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, criterion, DEVICE, optimizer)
        with torch.no_grad():
            val_loss = run_epoch(model, val_loader, criterion, DEVICE, None)
        history.append((epoch, train_loss, val_loss))
        print(f'Epoch {epoch:03d}/{EPOCHS} | train={train_loss:.6f} | val={val_loss:.6f}')

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f'Early stop at epoch {epoch}.')
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        _, z = model(torch.from_numpy(x_scaled.astype(np.float32)).to(DEVICE))
        z = z.cpu().numpy()

    history_df = pd.DataFrame(history, columns=['epoch', 'train_loss', 'val_loss'])
    return z, history_df, x_mean, x_std

In [ ]:
if RUN_AE_TRAINING:
    z, hist_df, x_mean, x_std = run_autoencoder(expr_hvg, labels)

    ae_df = pd.DataFrame(z, columns=[f'z{i+1}' for i in range(z.shape[1])])
    ae_df.insert(0, 'cell', cell_ids.values)
    ae_df['label'] = labels.values
    ae_df.to_csv(OUTDIR / 'autoencoder_latent.csv', index=False)
    hist_df.to_csv(OUTDIR / 'train_history.csv', index=False)
    pd.DataFrame({'mean': x_mean.ravel(), 'std': x_std.ravel()}, index=hvg_names).to_csv(OUTDIR / 'feature_scaler.csv')

    plt.figure(figsize=(7, 5))
    plt.plot(hist_df['epoch'], hist_df['train_loss'], label='train')
    plt.plot(hist_df['epoch'], hist_df['val_loss'], label='val')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Autoencoder Training Curve')
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    if z.shape[1] >= 2:
        plot_embedding(z[:, :2], labels, 'Autoencoder latent (first 2 dims)', 'z1', 'z2')
else:
    print('RUN_AE_TRAINING=False, skipped AE training.')

## Module F: Output Summary

In [ ]:
print('Output directory:', OUTDIR)
for p in sorted(OUTDIR.glob('*')):
    print('-', p.name)